# Laptop Inventory Dashboard

This notebook analyzes the laptop inventory in `pratice file.xlsx` and produces a dashboard showing:
- Brand distribution (Dell, Lenovo, HP, Apple)
- RAM capacity distribution
- Brand x Capacity matrix
- Status breakdown

**Author:** Fidelis Anyanwu  
**Date:** 2026-05-08

## 1. Setup and Imports

In [ ]:
import pandas as pd
import re
import matplotlib.pyplot as plt

# Make plots look nicer
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

## 2. Load the Excel File

In [ ]:
FILE_PATH = r'C:\Users\FidelisAnyanwu\OneDrive - M-KOPA\Desktop\pratice file\pratice file.xlsx'

df = pd.read_excel(FILE_PATH)
print(f'Rows: {df.shape[0]}, Columns: {df.shape[1]}')
print(f'Columns: {list(df.columns)}')
df.head()

## 3. Inspect the Raw Data

The `Make` column has the brand and the `Type` column has the model + RAM, e.g. `5320 i7 16gb`.

In [ ]:
print('Unique Make values (raw):')
print(df['Make'].value_counts(dropna=False))
print()
print('Sample Type values:')
print(df['Type'].dropna().unique()[:15])

## 4. Clean the Data

Two issues to fix:
1. **Brand casing inconsistencies** — `Dell`, `DELL`, `dell` should all be treated as one.
2. **RAM extraction** from the free-text `Type` column, plus a known typo (`i6gb` -> `16gb`).

In [ ]:
# 4a. Normalize brand names
df['Brand'] = df['Make'].astype(str).str.strip().str.upper().replace({'NAN': 'Unknown'})

# 4b. Extract RAM capacity from the Type column
def extract_ram(t):
    if pd.isna(t):
        return 'Not specified'
    s = str(t).lower()
    # Fix common typo: 'i6gb' -> '16gb'
    s = s.replace('i6gb', '16gb')
    # Find patterns like '8gb', '16 gb' etc., restricted to plausible RAM sizes
    matches = re.findall(r'\b(\d{1,2})\s*gb\b', s)
    valid = [m for m in matches if int(m) in [2, 4, 6, 8, 12, 16, 24, 32, 64]]
    if valid:
        return f'{valid[-1]}GB'
    # Fall back to e.g. '16g' (no trailing b)
    m = re.search(r'\b(\d{1,2})\s*g\b', s)
    if m and int(m.group(1)) in [4, 6, 8, 16, 32]:
        return f'{m.group(1)}GB'
    return 'Not specified'

df['RAM'] = df['Type'].apply(extract_ram)
df[['Make', 'Brand', 'Type', 'RAM']].head(10)

## 5. Dashboard - Brand Distribution

In [ ]:
brand_counts = df['Brand'].value_counts()
brand_pct = (brand_counts / len(df) * 100).round(1)

brand_summary = pd.DataFrame({'Count': brand_counts, '%': brand_pct})
brand_summary

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#4C72B0', '#DD8452', '#55A467', '#C44E52']

# Bar chart
brand_counts.plot(kind='bar', ax=ax1, color=colors)
ax1.set_title('Laptops by Brand', fontsize=13, fontweight='bold')
ax1.set_xlabel('Brand')
ax1.set_ylabel('Count')
for i, v in enumerate(brand_counts.values):
    ax1.text(i, v + 2, str(v), ha='center', fontweight='bold')
ax1.tick_params(axis='x', rotation=0)

# Pie chart
ax2.pie(brand_counts.values, labels=brand_counts.index, autopct='%1.1f%%',
        colors=colors, startangle=90)
ax2.set_title('Brand Share (%)', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

## 6. Dashboard - RAM Capacity Distribution

In [ ]:
ram_order = ['4GB', '6GB', '8GB', '16GB', '32GB', 'Not specified']
ram_counts = df['RAM'].value_counts().reindex(ram_order).dropna().astype(int)
ram_pct = (ram_counts / len(df) * 100).round(1)

ram_summary = pd.DataFrame({'Count': ram_counts, '%': ram_pct})
ram_summary

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ram_colors = ['#E74C3C', '#E67E22', '#F1C40F', '#27AE60', '#2980B9', '#95A5A6']
ram_counts.plot(kind='bar', ax=ax, color=ram_colors[:len(ram_counts)])
ax.set_title('Laptops by RAM Capacity', fontsize=13, fontweight='bold')
ax.set_xlabel('RAM')
ax.set_ylabel('Count')
for i, v in enumerate(ram_counts.values):
    ax.text(i, v + 2, str(v), ha='center', fontweight='bold')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

## 7. Dashboard - Brand x RAM Capacity Matrix

In [ ]:
pivot = pd.crosstab(df['Brand'], df['RAM'], margins=True, margins_name='TOTAL')
ordered_cols = [c for c in ram_order if c in pivot.columns] + ['TOTAL']
pivot = pivot[ordered_cols]
pivot

In [ ]:
# Stacked bar chart: brand on x-axis, RAM stacked
matrix = pd.crosstab(df['Brand'], df['RAM'])
matrix = matrix.reindex(columns=[c for c in ram_order if c in matrix.columns])

fig, ax = plt.subplots(figsize=(11, 6))
matrix.plot(kind='bar', stacked=True, ax=ax,
            color=ram_colors[:matrix.shape[1]])
ax.set_title('Brand x RAM Capacity', fontsize=13, fontweight='bold')
ax.set_xlabel('Brand')
ax.set_ylabel('Count')
ax.legend(title='RAM', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

## 8. Dashboard - Status Breakdown

In [ ]:
# Normalize status casing first
df['Status_clean'] = df['Status'].fillna('In Use / Unspecified').astype(str).str.strip().str.title()
status_counts = df['Status_clean'].value_counts()
status_counts

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
status_counts.plot(kind='barh', ax=ax, color='#3498DB')
ax.set_title('Laptops by Status', fontsize=13, fontweight='bold')
ax.set_xlabel('Count')
for i, v in enumerate(status_counts.values):
    ax.text(v + 1, i, str(v), va='center', fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Key Findings

- **316 laptops** in inventory.
- **Lenovo (48%) and Dell (47%)** dominate — together ~95% of the fleet.
- **8GB** is the most common explicitly-specified RAM (38%).
- **Lenovo skews lower** — 75% of Lenovos are 8GB.
- **Dell skews higher** — most specified Dells are 16GB or 32GB.
- **125 entries (~40%) have no RAM** listed in the source `Type` column, mostly Dells (e.g. `Latitude 7420`).
- One known typo (`i6gb`) was corrected to `16gb` during cleaning.